# Pipeline Bronze to Silver - Processamento de Dados

## Validando a SparkSession

In [0]:
spark

SparkSession - hive 
 
 
 SparkContext 

 Spark UI 

 
 Version 
 v3.3.2 
 Master 
 local[8] 
 AppName 
 Databricks Shell

## Configurações iniciais - Azure ADLS Gen2

In [0]:
storageAccountName = "datalake7eadf73a479de9f7"
sasToken = "sv=2024-11-04&ss=bfqt&srt=sco&sp=rwdlacupyx&se=2025-06-24T07:28:34Z&st=2025-06-23T23:28:34Z&spr=https&sig=idw%2B%2FOotMzKptQ8X5oECtDziKZ8sgPZM9CvPHeuOhqA%3D"

def mount_adls(blobContainerName):
    mount_point = f"/mnt/{storageAccountName}/{blobContainerName}"
    
    # Verificar se já está montado
    existing_mounts = [mount.mountPoint for mount in dbutils.fs.mounts()]
    
    if mount_point in existing_mounts:
        print(f"AVISO: {mount_point} já está montado. Desmontando primeiro...")
        try:
            dbutils.fs.unmount(mount_point)
            print(f"Desmontado com sucesso: {mount_point}")
        except Exception as e:
            print(f"Erro ao desmontar {mount_point}: {e}")
            return False
    
    # Montar o container
    try:
        dbutils.fs.mount(
            source = "wasbs://{}@{}.blob.core.windows.net".format(blobContainerName, storageAccountName),
            mount_point = mount_point,
            extra_configs = {'fs.azure.sas.' + blobContainerName + '.' + storageAccountName + '.blob.core.windows.net': sasToken}
        )
        print(f"Container {blobContainerName} montado com sucesso!")
        return True
    except Exception as e:
        print(f"Falha ao montar {blobContainerName}: {e}")
        return False

### Função para verificar se ambiente já está montado

In [0]:
def mount_adls_safe(blobContainerName):
    mount_point = f"/mnt/{storageAccountName}/{blobContainerName}"
    existing_mounts = [mount.mountPoint for mount in dbutils.fs.mounts()]
    
    if mount_point in existing_mounts:
        print(f"OK! {mount_point} já está montado.")
        return True
    else:
        return mount_adls(blobContainerName)

## Montando containers necessários

In [0]:
mount_adls('silver')

AVISO: /mnt/datalake7eadf73a479de9f7/silver já está montado. Desmontando primeiro...
/mnt/datalake7eadf73a479de9f7/silver has been unmounted.
Desmontado com sucesso: /mnt/datalake7eadf73a479de9f7/silver
Container silver montado com sucesso!
Out[23]: True

### Limpar container da camada Silver

In [0]:
def clean_silver_container():
    silver_path = f"/mnt/{storageAccountName}/silver"
    
    try:
        # Verificar se o path existe
        try:
            files_and_dirs = dbutils.fs.ls(silver_path)
        except Exception:
            print(f"Path {silver_path} não existe ou está vazio.")
            return True
        
        if not files_and_dirs:
            print("Container silver já está vazio.")
            return True
        
        print(f"Encontrados {len(files_and_dirs)} itens no silver para exclusão...")
        
        # Excluir cada item (arquivos e diretórios)
        for item in files_and_dirs:
            try:
                dbutils.fs.rm(item.path, True)  # True para recursivo
                print(f"✓ Excluído: {item.name}")
            except Exception as e:
                print(f"✗ Erro ao excluir {item.name}: {str(e)}")
                return False
        
        print("✅ Limpeza do container silver concluída com sucesso!")
        return True
        
    except Exception as e:
        print(f"❌ Erro geral na limpeza do silver: {str(e)}")
        return False

## Verificando containers montados

In [0]:
display(dbutils.fs.mounts())

mountPoint,source,encryptionType
/mnt/datalake7eadf73a479de9f7/gold,wasbs://gold@datalake7eadf73a479de9f7.blob.core.windows.net,
/databricks-datasets,databricks-datasets,
/mnt/datalake7eadf73a479de9f7/silver,wasbs://silver@datalake7eadf73a479de9f7.blob.core.windows.net,
/mnt/datalakefbca7dc4e981b9cb/landing-zone,wasbs://landing-zone@datalakefbca7dc4e981b9cb.blob.core.windows.net,
/databricks/mlflow-tracking,databricks/mlflow-tracking,sse-s3
/databricks-results,databricks-results,sse-s3
/mnt/datalakefbca7dc4e981b9cb/gold,wasbs://gold@datalakefbca7dc4e981b9cb.blob.core.windows.net,
/databricks/mlflow-registry,databricks/mlflow-registry,sse-s3
/mnt/datalake7eadf73a479de9f7/landing-zone,wasbs://landing-zone@datalake7eadf73a479de9f7.blob.core.windows.net,
/mnt/datalakefbca7dc4e981b9cb/silver,wasbs://silver@datalakefbca7dc4e981b9cb.blob.core.windows.net,


## Visualizando arquivos na camada Bronze

In [0]:
display(dbutils.fs.ls(f"/mnt/{storageAccountName}/bronze"))

path,name,size,modificationTime
dbfs:/mnt/datalake7eadf73a479de9f7/bronze/dbfs_excel_exports_achievement_unlocked_20250621_014612/,dbfs_excel_exports_achievement_unlocked_20250621_014612/,0,0
dbfs:/mnt/datalake7eadf73a479de9f7/bronze/dbfs_excel_exports_achievements_20250621_014608/,dbfs_excel_exports_achievements_20250621_014608/,0,0
dbfs:/mnt/datalake7eadf73a479de9f7/bronze/dbfs_excel_exports_developers_20250621_014548/,dbfs_excel_exports_developers_20250621_014548/,0,0
dbfs:/mnt/datalake7eadf73a479de9f7/bronze/dbfs_excel_exports_dlcs_20250621_014615/,dbfs_excel_exports_dlcs_20250621_014615/,0,0
dbfs:/mnt/datalake7eadf73a479de9f7/bronze/dbfs_excel_exports_game_genders_20250621_014619/,dbfs_excel_exports_game_genders_20250621_014619/,0,0
dbfs:/mnt/datalake7eadf73a479de9f7/bronze/dbfs_excel_exports_game_platforms_20250621_014625/,dbfs_excel_exports_game_platforms_20250621_014625/,0,0
dbfs:/mnt/datalake7eadf73a479de9f7/bronze/dbfs_excel_exports_game_tags_20250621_014622/,dbfs_excel_exports_game_tags_20250621_014622/,0,0
dbfs:/mnt/datalake7eadf73a479de9f7/bronze/dbfs_excel_exports_games_20250621_014544/,dbfs_excel_exports_games_20250621_014544/,0,0
dbfs:/mnt/datalake7eadf73a479de9f7/bronze/dbfs_excel_exports_genders_20250621_014554/,dbfs_excel_exports_genders_20250621_014554/,0,0
dbfs:/mnt/datalake7eadf73a479de9f7/bronze/dbfs_excel_exports_platforms_20250621_014551/,dbfs_excel_exports_platforms_20250621_014551/,0,0


## Importando bibliotecas necessárias

In [0]:
from pyspark.sql.functions import current_timestamp, lit

## Função para padronização de nomes de colunas

In [0]:
def padronizar_nomes_colunas(df, nome_tabela):
    """
    Padroniza nomes de colunas seguindo convenções:
    - Converte para maiúscula
    - Substitui prefixos por nomes completos
    - Adiciona metadados de processamento
    """
    
    # Renomear colunas seguindo padrões
    for coluna in df.columns:
        novo_nome = coluna.upper()
        
        # Substituições de prefixos
        novo_nome = novo_nome.replace("CD_", "CODIGO_")
        novo_nome = novo_nome.replace("VL_", "VALOR_")
        novo_nome = novo_nome.replace("DT_", "DATA_")
        novo_nome = novo_nome.replace("NM_", "NOME_")
        novo_nome = novo_nome.replace("DS_", "DESCRICAO_")
        novo_nome = novo_nome.replace("NR_", "NUMERO_")
        novo_nome = novo_nome.replace("_UF", "_UNIDADE_FEDERATIVA")
        
        df = df.withColumnRenamed(coluna, novo_nome)
    
    # Remover colunas de metadados antigos se existirem
    colunas_remover = ["DATA_HORA_BRONZE", "NOME_ARQUIVO"]
    for col in colunas_remover:
        if col in df.columns:
            df = df.drop(col)
    
    # Adicionar metadados da camada Silver
    df = df.withColumn("NOME_ARQUIVO_BRONZE", lit(nome_tabela))
    df = df.withColumn("DATA_HORA_SILVER", current_timestamp())
    
    return df

## Processamento Bronze to Silver - Versão Massiva

In [0]:
def process_bronze_to_silver():
    """Processa todas as tabelas da camada Bronze para Silver"""
    
    bronze_path = f"/mnt/{storageAccountName}/bronze"
    silver_path = f"/mnt/{storageAccountName}/silver"
    
    tabelas_processadas = {}
    
    try:
        bronze_dirs = dbutils.fs.ls(bronze_path)
        
        for dir_info in bronze_dirs:
            if dir_info.isDir():
                table_name = dir_info.name.rstrip('/')
                
                # Pular se não for uma tabela válida
                if not table_name or table_name.startswith('.'):
                    continue
                    
                print(f"\n=== Processando tabela: {table_name} ===")
                
                try:
                    # Carregar dados da camada Bronze
                    df_bronze = spark.read.format('delta').load(dir_info.path)
                    registro_count = df_bronze.count()
                    print(f"Registros carregados: {registro_count}")
                    
                    if registro_count == 0:
                        print(f"⚠️ Tabela {table_name} está vazia, pulando...")
                        continue
                    
                    # Aplicar transformações para Silver
                    df_silver = padronizar_nomes_colunas(df_bronze, table_name)
                    
                    # Criar nome limpo para Silver (remover prefixos longos)
                    silver_table_name = table_name.replace('dbfs_excel_exports_', '')
                    
                    # Salvar na camada Silver
                    df_silver.write.format('delta').mode('overwrite').save(f"{silver_path}/{silver_table_name}")
                    print(f"✓ Tabela {silver_table_name} salva na camada Silver")
                    
                    # Armazenar para validação
                    tabelas_processadas[silver_table_name] = df_silver
                    
                    # Mostrar esquema da tabela processada
                    print(f"Esquema da tabela {silver_table_name}:")
                    df_silver.printSchema()
                    
                except Exception as table_error:
                    print(f"✗ Erro ao processar tabela {table_name}: {str(table_error)}")
                    continue
                
    except Exception as e:
        print(f"Erro no processamento: {str(e)}")
    
    return tabelas_processadas

## Executando o processamento Bronze to Silver

In [0]:
silver_tables = process_bronze_to_silver()


=== Processando tabela: dbfs_excel_exports_achievement_unlocked_20250621_014612 ===
Registros carregados: 2000
✓ Tabela achievement_unlocked_20250621_014612 salva na camada Silver
Esquema da tabela achievement_unlocked_20250621_014612:
root
 |-- ID: integer (nullable = true)
 |-- UNLOCKDATE: timestamp (nullable = true)
 |-- USERID: integer (nullable = true)
 |-- ACHIEVEMENTID: integer (nullable = true)
 |-- NOME_ARQUIVO_BRONZE: string (nullable = false)
 |-- DATA_HORA_SILVER: timestamp (nullable = false)


=== Processando tabela: dbfs_excel_exports_achievements_20250621_014608 ===
Registros carregados: 200
✓ Tabela achievements_20250621_014608 salva na camada Silver
Esquema da tabela achievements_20250621_014608:
root
 |-- ID: integer (nullable = true)
 |-- NAME: string (nullable = true)
 |-- DESCRIPTION: string (nullable = true)
 |-- POINTS: integer (nullable = true)
 |-- GAMEID: integer (nullable = true)
 |-- NOME_ARQUIVO_BRONZE: string (nullable = false)
 |-- DATA_HORA_SILVER: time

## Validação dos dados processados na camada Silver

In [0]:
print("=== VALIDAÇÃO CAMADA SILVER ===")
display(dbutils.fs.ls(f"/mnt/{storageAccountName}/silver"))

=== VALIDAÇÃO CAMADA SILVER ===


path,name,size,modificationTime
dbfs:/mnt/datalake7eadf73a479de9f7/silver/achievement_unlocked_20250621_014612/,achievement_unlocked_20250621_014612/,0,0
dbfs:/mnt/datalake7eadf73a479de9f7/silver/achievements_20250621_014608/,achievements_20250621_014608/,0,0
dbfs:/mnt/datalake7eadf73a479de9f7/silver/developers_20250621_014548/,developers_20250621_014548/,0,0
dbfs:/mnt/datalake7eadf73a479de9f7/silver/dlcs_20250621_014615/,dlcs_20250621_014615/,0,0
dbfs:/mnt/datalake7eadf73a479de9f7/silver/game_genders_20250621_014619/,game_genders_20250621_014619/,0,0
dbfs:/mnt/datalake7eadf73a479de9f7/silver/game_platforms_20250621_014625/,game_platforms_20250621_014625/,0,0
dbfs:/mnt/datalake7eadf73a479de9f7/silver/game_tags_20250621_014622/,game_tags_20250621_014622/,0,0
dbfs:/mnt/datalake7eadf73a479de9f7/silver/games_20250621_014544/,games_20250621_014544/,0,0
dbfs:/mnt/datalake7eadf73a479de9f7/silver/genders_20250621_014554/,genders_20250621_014554/,0,0
dbfs:/mnt/datalake7eadf73a479de9f7/silver/platforms_20250621_014551/,platforms_20250621_014551/,0,0


## Função para validar tabelas Silver

In [0]:
def validar_tabelas_silver():
    """Valida as tabelas criadas na camada Silver"""
    
    silver_path = f"/mnt/{storageAccountName}/silver"
    
    try:
        silver_dirs = dbutils.fs.ls(silver_path)
        
        for dir_info in silver_dirs:
            if dir_info.isDir():
                table_name = dir_info.name.rstrip('/')
                print(f"\n=== Validação: {table_name} ===")
                
                df = spark.read.format('delta').load(dir_info.path)
                print(f"Total de registros: {df.count()}")
                print(f"Colunas: {df.columns}")
                
                # Mostrar primeiras linhas
                print(f"Primeiras 5 linhas da tabela {table_name}:")
                df.limit(5).display()
                
    except Exception as e:
        print(f"Erro na validação: {str(e)}")

## Executando validação das tabelas Silver

In [0]:
validar_tabelas_silver()


=== Validação: achievement_unlocked_20250621_014612 ===
Total de registros: 2000
Colunas: ['ID', 'UNLOCKDATE', 'USERID', 'ACHIEVEMENTID', 'NOME_ARQUIVO_BRONZE', 'DATA_HORA_SILVER']
Primeiras 5 linhas da tabela achievement_unlocked_20250621_014612:


ID,UNLOCKDATE,USERID,ACHIEVEMENTID,NOME_ARQUIVO_BRONZE,DATA_HORA_SILVER
2000,2025-06-17T09:38:18.742+0000,3,153,dbfs_excel_exports_achievement_unlocked_20250621_014612,2025-06-24T00:11:35.300+0000
2001,2025-06-18T00:02:03.351+0000,3,172,dbfs_excel_exports_achievement_unlocked_20250621_014612,2025-06-24T00:11:35.300+0000
2002,2025-06-17T06:11:10.028+0000,5,173,dbfs_excel_exports_achievement_unlocked_20250621_014612,2025-06-24T00:11:35.300+0000
2003,2025-06-17T14:38:30.227+0000,9,35,dbfs_excel_exports_achievement_unlocked_20250621_014612,2025-06-24T00:11:35.300+0000
2004,2025-06-17T08:21:25.971+0000,3,97,dbfs_excel_exports_achievement_unlocked_20250621_014612,2025-06-24T00:11:35.300+0000



=== Validação: achievements_20250621_014608 ===
Total de registros: 200
Colunas: ['ID', 'NAME', 'DESCRIPTION', 'POINTS', 'GAMEID', 'NOME_ARQUIVO_BRONZE', 'DATA_HORA_SILVER']
Primeiras 5 linhas da tabela achievements_20250621_014608:


ID,NAME,DESCRIPTION,POINTS,GAMEID,NOME_ARQUIVO_BRONZE,DATA_HORA_SILVER
1,stump hm,Quaerat a facere ab reiciendis accusamus ullam molestias.,67,5,dbfs_excel_exports_achievements_20250621_014608,2025-06-24T00:11:42.482+0000
2,infatuated underneath,Voluptas quae nemo esse odio.,28,10,dbfs_excel_exports_achievements_20250621_014608,2025-06-24T00:11:42.482+0000
3,mozzarella an,Ex praesentium fugit odit repudiandae magnam recusandae rem at.,67,2,dbfs_excel_exports_achievements_20250621_014608,2025-06-24T00:11:42.482+0000
4,thunderbolt knowledgeable,Ut saepe magnam eos beatae.,83,9,dbfs_excel_exports_achievements_20250621_014608,2025-06-24T00:11:42.482+0000
5,versus in,Cum dicta quibusdam et neque at suscipit corrupti cupiditate corrupti.,86,10,dbfs_excel_exports_achievements_20250621_014608,2025-06-24T00:11:42.482+0000



=== Validação: developers_20250621_014548 ===
Total de registros: 5
Colunas: ['ID', 'NAME', 'COUNTRY', 'NOME_ARQUIVO_BRONZE', 'DATA_HORA_SILVER']
Primeiras 5 linhas da tabela developers_20250621_014548:


...

ID,RATING,COMMENT,USERID,GAMEID,NOME_ARQUIVO_BRONZE,DATA_HORA_SILVER
1,2,Officia cupiditate quas veniam optio.,6,1,dbfs_excel_exports_reviews_20250621_014601,2025-06-24T00:12:45.988+0000
2,7,Tenetur assumenda in vitae eius iusto sit voluptates laborum.,6,4,dbfs_excel_exports_reviews_20250621_014601,2025-06-24T00:12:45.988+0000
3,3,Eaque natus doloribus autem molestiae aperiam culpa natus.,9,1,dbfs_excel_exports_reviews_20250621_014601,2025-06-24T00:12:45.988+0000
4,9,Officiis atque odio voluptatum minus animi porro praesentium illum odio.,7,10,dbfs_excel_exports_reviews_20250621_014601,2025-06-24T00:12:45.988+0000
5,2,Corrupti earum quos suscipit maxime.,4,3,dbfs_excel_exports_reviews_20250621_014601,2025-06-24T00:12:45.988+0000



=== Validação: tags_20250621_014558 ===
Total de registros: 4
Colunas: ['ID', 'NAME', 'NOME_ARQUIVO_BRONZE', 'DATA_HORA_SILVER']
Primeiras 5 linhas da tabela tags_20250621_014558:


ID,NAME,NOME_ARQUIVO_BRONZE,DATA_HORA_SILVER
1,provider,dbfs_excel_exports_tags_20250621_014558,2025-06-24T00:12:52.587+0000
2,department,dbfs_excel_exports_tags_20250621_014558,2025-06-24T00:12:52.587+0000
3,hope,dbfs_excel_exports_tags_20250621_014558,2025-06-24T00:12:52.587+0000
4,swath,dbfs_excel_exports_tags_20250621_014558,2025-06-24T00:12:52.587+0000



=== Validação: users_20250621_014541 ===
Total de registros: 10
Colunas: ['ID', 'NAME', 'EMAIL', 'PASSWORD', 'NOME_ARQUIVO_BRONZE', 'DATA_HORA_SILVER']
Primeiras 5 linhas da tabela users_20250621_014541:


ID,NAME,EMAIL,PASSWORD,NOME_ARQUIVO_BRONZE,DATA_HORA_SILVER
1,Fábio Oliveira,user0_Lucas.Macedo57@live.com,BMt56hy8NrPetuc,dbfs_excel_exports_users_20250621_014541,2025-06-24T00:12:58.463+0000
2,Caio Costa,user1_Beatriz.Santos51@gmail.com,W0d9cpIYAOOyabz,dbfs_excel_exports_users_20250621_014541,2025-06-24T00:12:58.463+0000
3,Sílvia Moreira,user2_Helena.Barros47@gmail.com,BaulET57063WWUZ,dbfs_excel_exports_users_20250621_014541,2025-06-24T00:12:58.463+0000
4,Rafaela Moraes,user3_Maite_Silva14@hotmail.com,keKoKdVE0YbM5Pm,dbfs_excel_exports_users_20250621_014541,2025-06-24T00:12:58.463+0000
5,Marli Santos,user4_Calebe_Moreira@hotmail.com,5HhaX_2hT__Oaoj,dbfs_excel_exports_users_20250621_014541,2025-06-24T00:12:58.463+0000


## Criação de tabelas Delta gerenciadas na camada Silver

In [0]:
def create_silver_managed_tables():
    """Cria tabelas Delta gerenciadas para a camada Silver"""
    
    database_name = "pipeline_silver"
    spark.sql(f"CREATE DATABASE IF NOT EXISTS {database_name}")
    
    silver_path = f"/mnt/{storageAccountName}/silver"
    
    try:
        silver_dirs = dbutils.fs.ls(silver_path)
        
        for dir_info in silver_dirs:
            if dir_info.isDir():
                table_name = dir_info.name.rstrip('/')
                
                # Pular diretórios inválidos
                if not table_name or table_name.startswith('.'):
                    continue
                
                try:
                    # Ler dados da camada Silver
                    df = spark.read.format('delta').load(dir_info.path)
                    
                    # Criar tabela gerenciada
                    df.write.format('delta').mode('overwrite').saveAsTable(f"{database_name}.{table_name}")
                    print(f"✓ Tabela gerenciada criada: {database_name}.{table_name}")
                    
                except Exception as table_error:
                    print(f"✗ Erro ao criar tabela {table_name}: {str(table_error)}")
                    continue
                
    except Exception as e:
        print(f"Erro ao criar tabelas gerenciadas: {str(e)}")

## Executando criação de tabelas gerenciadas

In [0]:
create_silver_managed_tables()

✓ Tabela gerenciada criada: pipeline_silver.achievement_unlocked_20250621_014612
✓ Tabela gerenciada criada: pipeline_silver.achievements_20250621_014608
✓ Tabela gerenciada criada: pipeline_silver.developers_20250621_014548
✓ Tabela gerenciada criada: pipeline_silver.dlcs_20250621_014615
✓ Tabela gerenciada criada: pipeline_silver.game_genders_20250621_014619
✓ Tabela gerenciada criada: pipeline_silver.game_platforms_20250621_014625
✓ Tabela gerenciada criada: pipeline_silver.game_tags_20250621_014622
✓ Tabela gerenciada criada: pipeline_silver.games_20250621_014544
✓ Tabela gerenciada criada: pipeline_silver.genders_20250621_014554
✓ Tabela gerenciada criada: pipeline_silver.platforms_20250621_014551
✓ Tabela gerenciada criada: pipeline_silver.purchases_20250621_014605
✓ Tabela gerenciada criada: pipeline_silver.reviews_20250621_014601
✓ Tabela gerenciada criada: pipeline_silver.tags_20250621_014558
✓ Tabela gerenciada criada: pipeline_silver.users_20250621_014541


## Verificação das tabelas Silver gerenciadas

In [0]:
print("=== TABELAS SILVER GERENCIADAS ===")
spark.sql("SHOW TABLES IN pipeline_silver").show()

=== TABELAS SILVER GERENCIADAS ===
+---------------+--------------------+-----------+
|       database|           tableName|isTemporary|
+---------------+--------------------+-----------+
|pipeline_silver|achievement_unlocked|      false|
|pipeline_silver|achievement_unloc...|      false|
|pipeline_silver|        achievements|      false|
|pipeline_silver|achievements_2025...|      false|
|pipeline_silver|          developers|      false|
|pipeline_silver|developers_202506...|      false|
|pipeline_silver|                dlcs|      false|
|pipeline_silver|dlcs_20250621_014615|      false|
|pipeline_silver|        game_genders|      false|
|pipeline_silver|game_genders_2025...|      false|
|pipeline_silver|      game_platforms|      false|
|pipeline_silver|game_platforms_20...|      false|
|pipeline_silver|           game_tags|      false|
|pipeline_silver|game_tags_2025062...|      false|
|pipeline_silver|               games|      false|
|pipeline_silver|games_20250621_01...|      fal

## Função para análise de qualidade dos dados

In [0]:
def analise_qualidade_silver():
    """Realiza análise básica de qualidade dos dados na camada Silver"""
    
    database_name = "pipeline_silver"
    
    try:
        # Listar todas as tabelas
        tabelas = spark.sql(f"SHOW TABLES IN {database_name}").collect()
        
        for tabela in tabelas:
            nome_tabela = tabela['tableName']
            print(f"\n=== Análise de Qualidade: {nome_tabela} ===")
            
            # Contagem total de registros
            total_registros = spark.sql(f"SELECT COUNT(*) as total FROM {database_name}.{nome_tabela}").collect()[0]['total']
            print(f"Total de registros: {total_registros}")
            
            # Verificar valores nulos por coluna
            df = spark.sql(f"SELECT * FROM {database_name}.{nome_tabela}")
            
            print("Valores nulos por coluna:")
            for coluna in df.columns:
                if coluna not in ['NOME_ARQUIVO_BRONZE', 'DATA_HORA_SILVER']:
                    nulos = df.filter(df[coluna].isNull()).count()
                    percentual = (nulos / total_registros) * 100 if total_registros > 0 else 0
                    print(f"  {coluna}: {nulos} ({percentual:.2f}%)")
            
    except Exception as e:
        print(f"Erro na análise de qualidade: {str(e)}")

## Executando análise de qualidade

In [0]:
analise_qualidade_silver()


=== Análise de Qualidade: achievement_unlocked ===
Total de registros: 2
Valores nulos por coluna:
  ID: 0 (0.00%)
  USER_ID: 0 (0.00%)
  ACHIEVEMENT_ID: 0 (0.00%)

=== Análise de Qualidade: achievement_unlocked_20250621_014612 ===
Total de registros: 2000
Valores nulos por coluna:
  ID: 0 (0.00%)
  UNLOCKDATE: 0 (0.00%)
  USERID: 0 (0.00%)
  ACHIEVEMENTID: 0 (0.00%)

=== Análise de Qualidade: achievements ===
Total de registros: 2
Valores nulos por coluna:
  ID: 0 (0.00%)
  GAME_ID: 0 (0.00%)
  TITLE: 0 (0.00%)
  DESCRIPTION: 0 (0.00%)

=== Análise de Qualidade: achievements_20250621_014608 ===
Total de registros: 200
Valores nulos por coluna:
  ID: 0 (0.00%)
  NAME: 0 (0.00%)
  DESCRIPTION: 0 (0.00%)
  POINTS: 0 (0.00%)
  GAMEID: 0 (0.00%)

=== Análise de Qualidade: developers ===
Total de registros: 2
Valores nulos por coluna:
  ID: 0 (0.00%)
  NAME: 0 (0.00%)

=== Análise de Qualidade: developers_20250621_014548 ===
Total de registros: 5
Valores nulos por coluna:
  ID: 0 (0.00%)
 